#### Kuka
The authoring, visualization and simulation front end run as a service, as a scriptable python module called pyaraas. We also use the pyaraas module to script the assembly behaviors. We've setup the scripting environment to run independently of the authoring tool. This means you can run your script from whatever environment you like, an IDE, a repl, a script file, or even from a notebook. 

When running this notebook, make sure it's running the python environment (3.7) where the pyaraas module is installed.

Launch the araas tool and use the default Sample project.

Here we load the Kuka workcell and get a handle to the gripper attached to the Kuka robot.

In [1]:
import pyaraas
from pyaraas import Transform
from pyaraas import Task

workcell = pyaraas.start()
gripper = workcell.get_gripper("SchunkDual-0")

#### async/await
Commanding robots are asynchronous in nature. You request them to do some task, and then some time later, hopefully the task is completed or some report of failure is received. Under the hood we use the [trio python module](https://github.com/python-trio/trio) to manage the asynchronous programming using async/await. In one sense it acts like a blocking call, but it runs in a manner that support simple to implement concurrency (commanding robots and peripherals to run at the same time).

Here we command the gripper to open it's finger to 80 mm separation in 5 seconds. On success (after 5 seconds), we get a 'finished' message from the call to tell us it succeeded. In the tool, you should see the fingers of the gripper slowly open to 80 mm.

In a notebook cell, calling `await gripper.open`, the notebook understands that `gripper.open` is an *asynchronous* function defined in the pyaras module, and by calling it with the keyword `await` in front of it, it runs `gripper.open` and *awaits* for the call to finish.

In [2]:
await gripper.open(80, 5)

'finished'

#### run
In other environments like running behaviors from a script file, you will need to use `asyncio.run` or `trio.run` to run the asynchronous function. We've added a helper function to pyaraas (`pyaraas.run`) to make it easier to run an asynchronous task and multiple tasks concurrently. We recommend you use the `pyaraas.run` function instead of using `tri.run` and its `nurseries` feature, since our `run` also adds some monitoring functionality.

Here we show how to call `gripper.open` using the `pyaraas.run` function. Note, the odd manner in which you specify the asynchronous funcion you want to run, is a requirement of the trio module we use.

In [3]:
pyaraas.run(Task(gripper.open, 0, 5))

['finished']

#### concurrency
The benefit of using `pyaraas.run` is evident when you want to run tasks concurrently.

Here we run two tasks at the same time, opening the gripper and moving the gripper. You should see both tasks running concuurrenly on the robot/gripper. Note, the second task is done in 2 seconds, but because the opening fingers takes 5 seconds, the task is *awaited* for the full 5 seconds (waiting for both tasks to be done).

In [4]:
task1 = Task(gripper.open, 100, 5)
task2 = Task(gripper.move_to, Transform(3021.35,0.00,1011.39,0.00000,-1.35891,0.00000), 2)
pyaraas.run([task1, task2])

['finished', 'finished']

#### async function
To build more complicated tasks, you can define an async function that contains the multiple steps. For any asynchronous function that you call within your function, be sure to add the `await` keyword to notify python to wait for the call to complete before executing the next step. To add concurrent subtasks within your function use the `pyaraas.gather` helper function to specify the tasks that should run concurrrently.

In this example, we define and run a multi step task that moves the gripper to a specified location, then closes the gripper and moves it to another location concurrently. 

In [5]:
async def multi_step_task(gripper):
    await gripper.move_to(Transform(2360.02,-0.00,436.67,0.00000,-0.59376,0.00000), 5)
    await pyaraas.gather([Task(gripper.open, 0, 5), 
                          Task(gripper.move_to, Transform(2360.02,-0.00,1436.67,0.00000,-0.59376,0.00000), 5)])

pyaraas.run(Task(multi_step_task, gripper))

[None]

#### shutting down
When you start a scripting session, the connected workcell comes online and is considered *live*. Once the assembly behavior is done, we can disconnect from the workcell by calling `pyaraas.stop()`.

In [6]:
pyaraas.stop()

scripting session closed
araas session closed
